# SAT4J

[SAT4J](https://www.sat4j.org) is a SAT solver that is implemented in Java.  It is based on the ideas of the SAT solver [MiniSat](http://minisat.se).  SAT4J is available from [Maven Central](https://central.sonatype.com/artifact/org.ow2.sat4j/org.ow2.sat4j.core), which is the standard repository for Java libraries.  The IJava kernel can download a library from Maven Central via the magic command `%maven`.  The first time this command is executed, it needs a connection to the internet.

In [ ]:
%maven org.ow2.sat4j:org.ow2.sat4j.core:2.3.6

We import the classes and interfaces of SAT4J that we need.  As we do not use the class `RecursiveSet` in this notebook, we also import the package `java.util.stream`.

Note that the IJava kernel imports the package `java.util.concurrent` by default.  This package contains a class `TimeoutException`, and so does the package `org.sat4j.specs`.  To avoid an ambiguity, we import the classes from `org.sat4j.specs` explicitly, since an explicit import takes precedence over an import of the form `import java.util.concurrent.*;`.

In [ ]:
import java.util.stream.*;
import org.sat4j.core.VecInt;
import org.sat4j.minisat.SolverFactory;
import org.sat4j.specs.ContradictionException;
import org.sat4j.specs.ISolver;
import org.sat4j.specs.TimeoutException;

In this notebook, the SAT solver [SAT4J](https://www.sat4j.org) is used to efficiently solve propositional logic (SAT) problems.

- *Propositional variables* are represented by positive natural numbers, i.e. by `1`, `2`, `3`, $\cdots$.
- *Negations* are represented by a preceding minus sign (e.g., `-2` for $\neg x_2$).
- *Clauses* are represented as objects of the class `VecInt`, which is a vector of integers.
  The expression `new VecInt(new int[] {1, -2, 3})` creates the clause $\{ x_1, \neg x_2, x_3 \}$.
- A *solver* is created via:
  ```java
  ISolver solver = SolverFactory.newDefault();
  ```
- The clauses describing the problem are added to the solver using the method
  `solver.addClause(...)`.  The set of all clauses corresponds to a large conjunction of all clauses.
- Besides clauses, SAT4J also supports *cardinality constraints*:
  - `solver.addAtMost(literals, k)` specifies that at most `k` of the given literals are true,
  - `solver.addAtLeast(literals, k)` specifies that at least `k` of the given literals are true,
  - `solver.addExactly(literals, k)` specifies that exactly `k` of the given literals are true.
- The satisfiability check is performed using `solver.isSatisfiable()`.
- If the problem is solvable, `solver.model()` returns the solution as an array of literals:
  A positive literal `i` means that the variable $x_i$ is `true`, a negative literal `-i` means that $x_i$ is `false`.

The methods that add clauses or constraints may throw a `ContradictionException` if the solver detects that the problem is trivially unsatisfiable.  The method `isSatisfiable` may throw a `TimeoutException`.  As these are *checked exceptions*, methods calling these methods have to declare them in their `throws` clause.

As an example, the formula
$$ (\neg p \vee  q \vee \neg r) \wedge (p \vee \neg q \vee r) \wedge (p \vee q \vee \neg r) \wedge (p \vee q \vee r), $$
which is in conjunctive normal form, is represented by the following code, where the variables $p$, $q$, and $r$ are represented by the numbers `1`, `2`, and `3`:

In [ ]:
ISolver solver = SolverFactory.newDefault();
int[][] clauses = { {-1,  2, -3},
                    { 1, -2,  3},
                    { 1,  2, -3},
                    { 1,  2,  3} };
for (int[] clause : clauses) {
    solver.addClause(new VecInt(clause));
}

In [ ]:
solver.isSatisfiable()

In [ ]:
Arrays.toString(solver.model())

This shows that the formula `f` is satisfiable and that the propositional valuation
$$ \mathcal{I} = \{ p \mapsto \texttt{False}, q \mapsto \texttt{True}, r \mapsto \texttt{True} \} $$
is a solution for `f`, i.e. we have
$$ \mathcal{I}(\texttt{f}) = \texttt{True}. $$
Note that the formula has other solutions, too.  Hence a different version of SAT4J might return a different solution.

## Sudoku

As SAT4J uses numbers to represent variables, while we want to use readable names like `Q<1,2,3>`, we need a mapping between names and numbers.  The function `variable(name)` returns the number of the variable with the given name.  If this name has not been used before, a new number is assigned to it.  The function `variableName(number)` returns the name of the variable with the given number.

In [ ]:
Map<String, Integer> variableNumbers = new HashMap<>();
List<String>         variableNames   = new ArrayList<>();

In [ ]:
int variable(String name) {
    return variableNumbers.computeIfAbsent(name, n -> {
        variableNames.add(n);
        return variableNames.size();
    });
}

String variableName(int number) {
    return variableNames.get(number - 1);
}

The function `range` creates the list $[ \texttt{start}, \cdots, \texttt{stop} ]$.

In [ ]:
List<Integer> range(int start, int stop) {
    return IntStream.rangeClosed(start, stop).boxed().toList();
}

The finish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is shown below.

<img src="sudoku.png" alt="A game of Sudoku" width="50%">

The function `createPuzzle` returns the Sudoku as a two-dimensional array.  This array contains the number `0` for those cells that are left unspecified.

In [ ]:
int[][] createPuzzle() {
    return new int[][] {
        { 8, 0, 0, 0, 0, 0, 0, 0, 0 },
        { 0, 0, 3, 6, 0, 0, 0, 0, 0 },
        { 0, 7, 0, 0, 9, 0, 2, 0, 0 },
        { 0, 5, 0, 0, 0, 7, 0, 0, 0 },
        { 0, 0, 0, 0, 4, 5, 7, 0, 0 },
        { 0, 0, 0, 1, 0, 0, 0, 3, 0 },
        { 0, 0, 1, 0, 0, 0, 0, 6, 8 },
        { 0, 0, 8, 5, 0, 0, 0, 1, 0 },
        { 0, 9, 0, 0, 0, 0, 4, 0, 0 }
    };
}

We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `varName(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [ ]:
String varName(int row, int col, int digit) {
    return "Q<" + row + "," + col + "," + digit + ">";
}

In [ ]:
varName(1, 2, 3)

The function `exactlyOne(solver, names)` takes a solver and a list of variable names.  It adds a cardinality constraint to the solver specifying that exactly one of these variables is `true`.

In [ ]:
void exactlyOne(ISolver solver, List<String> names) throws ContradictionException {
    int[] literals = names.stream().mapToInt(name -> variable(name)).toArray();
    solver.addExactly(new VecInt(literals), 1);
}

A `Position` is a pair of the form `(row, col)`, where `row` is the row
of a cell and `col` is its column.

In [ ]:
record Position(int row, int col) {
    public String toString() { return "(" + row + ", " + col + ")"; }
}

The function `exactlyOneForDigit` takes three arguments:
* A `solver`.
* A list `L` of positions representing Sudoku coordinates.
  The elements of `L` are pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
* A specific `digit`.  `digit` is an element from the set `{1,...,9}`.

It adds a constraint to the solver expressing the fact that the given `digit`
appears exactly once among the fields specified by the coordinate pairs in `L`.

In [ ]:
void exactlyOneForDigit(ISolver solver, List<Position> L, int digit) throws ContradictionException {
    exactlyOne(solver, L.stream().map(pos -> varName(pos.row(), pos.col(), digit)).toList());
}

The function `exactlyOneForEveryDigit` takes a solver and a list `L` of positions as its arguments.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It adds constraints expressing that all digits from the set
$\{1,\cdots,9\}$ occur exactly once in the list `L` of Sudoku cells.

In [ ]:
void exactlyOneForEveryDigit(ISolver solver, List<Position> L) throws ContradictionException {
    for (int d : range(1, 9)) {
        exactlyOneForDigit(solver, L, d);
    }
}

The function `exactlyOneDigit(solver, row, col)` takes a solver and integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function adds a constraint specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [ ]:
void exactlyOneDigit(ISolver solver, int row, int col) throws ContradictionException {
    exactlyOne(solver, range(1, 9).stream().map(d -> varName(row, col, d)).toList());
}

The function `constraintsFromPuzzle` adds unit clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [ ]:
void constraintsFromPuzzle(ISolver solver) throws ContradictionException {
    var Puzzle = createPuzzle();
    for (int r = 1; r <= 9; ++r) {
        for (int c = 1; c <= 9; ++c) {
            if (Puzzle[r-1][c-1] != 0) {
                int literal = variable(varName(r, c, Puzzle[r-1][c-1]));
                solver.addClause(new VecInt(new int[] { literal }));
            }
        }
    }
}

The function `getBlockCells` calculates the absolute grid coordinates for all 9 cells 
within a specific 3x3 Sudoku block.  The Sudoku board is divided into a 3x3 grid of blocks. 
This function takes the zero-based indices of a block and maps its local cells to the 
1-based coordinates used by the solver variables. 
 * `r` - The row index of the 3x3 macro-block (must be 0, 1, or 2).
 * `c` - The column index of the 3x3 macro-block (must be 0, 1, or 2).
The function returns a list of 9 positions, where each position `(row, col)` represents 
 the 1-based row and column coordinates on the full 9x9 Sudoku board.
 
For example, the set of coordinates for the top-middle block (`r = 0, c = 1`) 
is given as follows:
```getBlockCells(0, 1) = 
         [[1, 4], [1, 5], [1, 6], 
          [2, 4], [2, 5], [2, 6], 
          [3, 4], [3, 5], [3, 6]]
```


In [ ]:
List<Position> getBlockCells(int r, int c) {
    List<Position> Result = new ArrayList<>();
    for (int row = 1; row <= 3; ++row) {
        for (int col = 1; col <= 3; ++col) {
            Result.add(new Position(r * 3 + row, c * 3 + col));
        }
    }
    return Result;
}

In [ ]:
getBlockCells(0, 1)

The function `allConstraints` adds all constraints describing the given Sudoku to the solver and checks whether these constraints are satisfiable.

Note that inside the lambda expressions `col -> new Position(row, col)` and `row -> new Position(row, col)` we can only use variables that are *effectively final*, i.e. variables that are never changed.  Therefore, we use *for-each* loops instead of loops of the form `for (int row = 1; row <= 9; ++row)`, since the variable `row` would be changed by `++row`.

In [ ]:
boolean allConstraints(ISolver solver) throws ContradictionException, TimeoutException {
    // 1. Start with constraints from the puzzle
    constraintsFromPuzzle(solver);
    // 2. There is exactly one digit in every field
    for (int row : range(1, 9)) {
        for (int col : range(1, 9)) {
            exactlyOneDigit(solver, row, col);
        }
    }
    // 3. All entries in a row are unique
    for (int row : range(1, 9)) {
        exactlyOneForEveryDigit(solver, range(1, 9).stream().map(col -> new Position(row, col)).toList());
    }
    // 4. All entries in a column are unique
    for (int col : range(1, 9)) {
        exactlyOneForEveryDigit(solver, range(1, 9).stream().map(row -> new Position(row, col)).toList());
    }
    // 5. All entries in a 3x3 square are unique
    for (int row : range(0, 2)) {
        for (int col : range(0, 2)) {
            exactlyOneForEveryDigit(solver, getBlockCells(row, col));
        }
    }
    return solver.isSatisfiable();
}

In [ ]:
ISolver solver = SolverFactory.newDefault();
long start = System.currentTimeMillis();
boolean solvable = allConstraints(solver);
System.out.println("solve: " + (System.currentTimeMillis() - start) + " ms");
solvable

SAT4J_TIME_PLACEHOLDER

The function `getTrueVars` returns the names of those variables that are `true` in the solution found by the solver.

In [ ]:
List<String> getTrueVars(ISolver solver) {
    return Arrays.stream(solver.model())
                 .filter(literal -> literal > 0)
                 .mapToObj(literal -> variableName(literal))
                 .toList();
}

In [ ]:
var Variables = getTrueVars(solver);
Variables

## Graphical Representation

The function `showSolution` displays the solution as an HTML table.  The digits that are given in the puzzle are shown in bold.  The names of the variables have the form `Q<r,c,d>`.  In order to extract `r`, `c`, and `d`, we strip the first two characters and the last character of a variable name and split the remaining string at the commas.

In [ ]:
void showSolution(List<String> Solution, String width) {
    var Sudoku   = createPuzzle();
    var solution = new int[9][9];
    for (var name : Solution) {
        var parts = name.substring(2, name.length() - 1).split(",");
        int row   = Integer.parseInt(parts[0]);
        int col   = Integer.parseInt(parts[1]);
        int digit = Integer.parseInt(parts[2]);
        solution[row - 1][col - 1] = digit;
    }
    var html = new StringBuilder();
    html.append("""
        <table style="width:%s; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;">
        """.formatted(width));
    for (int row = 0; row < 9; row++) {
        html.append("<tr>");
        for (int col = 0; col < 9; col++) {
            int    value     = solution[row][col];
            String cellStyle = "font-weight: normal;";
            if (Sudoku[row][col] != 0) {
                value     = Sudoku[row][col];
                cellStyle = "font-weight: bold;";
            }
            boolean isGray      = (row / 3 + col / 3) % 2 != 0;
            String  bgColor     = isGray ? "#f0f0f0" : "#ffffff";
            String  borderStyle = "border: 1px solid #ccc;";
            if ((col + 1) % 3 == 0 && col < 8) borderStyle += "border-right: 2px solid black;";
            if ((row + 1) % 3 == 0 && row < 8) borderStyle += "border-bottom: 2px solid black;";
            html.append("""
                <td style="%s width:30px; height:30px; text-align:center; font-size:20px; \
                background-color:%s; %s">%s</td>""".formatted(
                    borderStyle, bgColor, cellStyle, value != 0 ? value : ""));
        }
        html.append("</tr>");
    }
    html.append("</table>");
    display(html.toString(), "text/html");
}

In [ ]:
showSolution(Variables, "50%");